# The Price is Right

### Week 7 Day 2

Selecting our model and evaluating the base model against the task.

Keep in mind: our base model has 8 billion params, quantized down to 4 bits
Compared with GPT-5 at TRILLIONS of params!

In [466]:
# ! python -m pip install transformers accelerate bitsandbytes peft trl unsloth datasets --upgrade

In [467]:
import os

# Set env vars FIRST
os.environ["HF_HOME"] = "/tmp/mawojide/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/mawojide/hf_cache/transformers"
os.environ["HF_DATASETS_CACHE"] = "/tmp/mawojide/hf_cache/datasets"

os.makedirs("/tmp/mawojide/hf_cache/transformers", exist_ok=True)
os.makedirs("/tmp/mawojide/hf_cache/datasets", exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])  # confirm it's set

# THEN import and verify
from huggingface_hub import constants
print(constants.HF_HUB_CACHE)   # should show /tmp/mawojide/hf_cache/hub

HF_HOME: /tmp/mawojide/hf_cache
/tmp/mawojide/hf_cache/hub


In [468]:
! wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [469]:
# imports

import os
import re
import math
from tqdm import tqdm
# from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
import matplotlib.pyplot as plt
# from util import evaluate

In [470]:
# Constants
# BASE_MODEL = "unsloth/Qwen2.5-Math-7B" # - 146(NF4) 129(FP8)
# BASE_MODEL = "Qwen/Qwen3-14B-Base" # - 160 (NF4) 141(FP8)
# BASE_MODEL = "meta-llama/Llama-3.2-3B" # - 147(NF4) 133(FP8)
# BASE_MODEL = "unsloth/Qwen2.5-7B-bnb-4bit" - 180
# BASE_MODEL = "Qwen/Qwen2.5-Math-72B"
BASE_MODEL = "Qwen/Qwen2.5-Math-1.5B" # - 114(NF4) 162(FP8)

PROJECT_NAME = "price"

LITE_MODE = True
# LITE_MODE = False

DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# Hyper-parameters

QUANT_4_BIT = True
# QUANT_4_BIT = False

### Log in to HuggingFace

If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon in the left, and add a new secret called `HF_TOKEN` with the value as your token.

In [471]:
# Log in to HuggingFace

# hf_token = userdata.get('HF_TOKEN')
hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Load our data

We uploaded it to Hugging Face, so it's easy to retrieve it now

In [472]:
# dataset = load_dataset(DATASET_NAME, streaming=True)
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val']
test = dataset['test']

In [473]:
train[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.\n\nPrice is $',
 'completion': '64.00'}

# Prepare our Base Llama Model for evaluation

Load our base model with 4 bit quantization and try out 1 example

In [474]:
## pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [475]:
# Load the Tokenizer and the Model
from transformers import AutoConfig

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the configuration and set pad_token_id if it's missing
config = AutoConfig.from_pretrained(BASE_MODEL)
# Ensure pad_token_id is set in the config, as StableLmModel expects it
config.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    config=config, # Pass the modified config
    quantization_config=quant_config,
    device_map="auto",
)

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Memory footprint: 1.1 GB


In [476]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = base_model.generate(
            **inputs,
            max_new_tokens=8,
            pad_token_id=tokenizer.eos_token_id # Explicitly pass pad_token_id
        )
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [477]:
import statistics

def extract_price(text):
    """Extract the first valid price from generated text."""
    patterns = [
        r'\$[\d,]+\.?\d*',
        r'[\d,]+\.\d{2}',
        r'[\d,]+',
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            price_str = match.group().replace('$', '').replace(',', '')
            try:
                price = float(price_str)
                if 0.01 <= price <= 100000:
                    return price
            except:
                continue
    return None

"""
def model_predict_ensemble(item, n_samples=10, temperature=0.7):
    # Generate n_samples predictions and return the median.
    prompt = str(item["prompt"])
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    prices = []
    with torch.no_grad():
        for _ in range(n_samples):
            output_ids = base_model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=True,
                temperature=temperature,
                pad_token_id=tokenizer.eos_token_id,
            )
            # Decode only the newly generated tokens
            new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
            generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            price = extract_price(generated_text)
            if price:
                prices.append(price)
    if not prices:
        return None
    return statistics.geometric_mean(prices)
"""

'\ndef model_predict_ensemble(item, n_samples=10, temperature=0.7):\n    # Generate n_samples predictions and return the median.\n    prompt = str(item["prompt"])\n    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")\n    prices = []\n    with torch.no_grad():\n        for _ in range(n_samples):\n            output_ids = base_model.generate(\n                **inputs,\n                max_new_tokens=8,\n                do_sample=True,\n                temperature=temperature,\n                pad_token_id=tokenizer.eos_token_id,\n            )\n            # Decode only the newly generated tokens\n            new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]\n            generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)\n            price = extract_price(generated_text)\n            if price:\n                prices.append(price)\n    if not prices:\n        return None\n    return statistics.geometric_mean(prices)\n'

In [ ]:
num_beams = 10
num_groups = 10
diversity_penalty = 1.5

def model_predict_diverse_beam(item, num_beams=num_beams, num_groups=num_groups, diversity_penalty=diversity_penalty):
    prompt = str(item["prompt"])
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = base_model.generate(
            **inputs,
            # max_new_tokens=8,
            num_beams=num_beams,
            num_beam_groups=num_groups,
            diversity_penalty=diversity_penalty,
            num_return_sequences=num_beams,
            pad_token_id=tokenizer.eos_token_id,
            trust_remote_code=True,
        )

    prices = []
    for beam_output in output_ids:
        text = tokenizer.decode(beam_output[input_length:], skip_special_tokens=True)
        price = extract_price(text)
        if price:
            prices.append(price)

    if not prices:
        return None
    return statistics.geometric_mean(prices)


In [479]:
# # Grid search for diverse beam search

# def evaluate_ensemble_diverse(data, num_beams=10, num_groups=5, diversity_penalty=1.0, size=100):
#     errors = []
#     for i in tqdm(range(size), desc=f"beams={num_beams} pen={diversity_penalty}"):
#         item = data[i]
#         pred = model_predict_diverse_beam(item, num_beams=num_beams, num_groups=num_groups, diversity_penalty=diversity_penalty)
#         if pred is None:
#             continue
#         truth = float(item["completion"])
#         errors.append(abs(pred - truth))
#     return sum(errors) / len(errors) if errors else float("inf")


# results = []
# for num_beams in [5, 10, 20]:
#     for num_groups in [2, 5, 10]:
#         if num_beams % num_groups != 0:
#             continue
#         for div_pen in [0.3, 0.5, 1.0, 1.5, 2.0]:
#             mae = evaluate_ensemble_diverse(test, num_beams=num_beams, num_groups=num_groups, diversity_penalty=div_pen)
#             results.append((mae, num_beams, num_groups, div_pen))

# # Print sorted by MAE (best first)
# for mae, nb, ng, dp in sorted(results):
#     print(f"beams={nb}, groups={ng}, div_pen={dp} → MAE={mae:.1f}")

In [480]:
test[0]

{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nPrice is $',
 'completion': '219.0'}

In [481]:
print(type(test[0]))
print(test[0])
print(type(test[0]["prompt"]))
print(repr(test[0]["prompt"]))  # repr shows None, NaN etc clearly

<class 'dict'>
{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.\n\nPrice is $', 'completion': '219.0'}
<class 'str'>
'What does this cost to the nearest dollar?\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per mo

In [482]:
model_predict(test[0])

'120.00. What'

# Evaluation!

Trying out our base Llama 3.2 model against the Test dataset

In [486]:
import re
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from itertools import accumulate
import math
from tqdm.auto import tqdm
from IPython.display import clear_output


GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

DEFAULT_SIZE = 200


class Tester:
    def __init__(self, predictor, data, title=None, size=DEFAULT_SIZE):
        self.predictor = predictor
        self.data = data
        self.title = title or self.make_title(predictor)
        self.size = size
        self.titles = []
        self.guesses = []
        self.truths = []
        self.errors = []
        self.colors = []

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title().replace("Gpt", "GPT")

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        else:
            return value

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:
            return "green"
        elif error < 80 or error / truth < 0.4:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value = self.predictor(datapoint)
        guess = self.post_process(value)
        truth = float(datapoint["completion"])
        error = abs(guess - truth)
        color = self.color_for(error, truth)
        pieces = datapoint["prompt"].split("Title: ")
        title = pieces[1].split("\n")[0] if len(pieces) > 1 else pieces[0]
        title = title if len(title) <= 40 else title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame(
            {
                "truth": self.truths,
                "guess": self.guesses,
                "title": self.titles,
                "error": self.errors,
                "color": self.colors,
            }
        )

        # Pre-format hover text
        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]

        max_val = float(max(df["truth"].max(), df["guess"].max()))

        fig = px.scatter(
            df,
            x="truth",
            y="guess",
            color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title,
            labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=800,
            height=600,
        )

        # Assign customdata per trace (one color/category = one trace)
        for tr in fig.data:
            mask = df["color"] == tr.name
            tr.customdata = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)

        # Reference line y=x
        fig.add_trace(
            go.Scatter(
                x=[0, max_val],
                y=[0, max_val],
                mode="lines",
                line=dict(width=2, dash="dash", color="deepskyblue"),
                name="y = x",
                hoverinfo="skip",
                showlegend=False,
            )
        )

        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n = len(self.errors)

        # Running mean and std (pure Python)
        running_sums = list(accumulate(self.errors))
        x = list(range(1, n + 1))
        running_means = [s / i for s, i in zip(running_sums, x)]

        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds = [
            math.sqrt((sq_sum / i) - (mean**2)) if i > 1 else 0
            for i, sq_sum, mean in zip(x, running_squares, running_means)
        ]

        # 95% confidence interval for mean
        ci = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]

        # Title with final stats
        final_mean = running_means[-1]
        final_ci = ci[-1]
        title = f"{self.title} Error: {final_mean:,.2f} ± {final_ci:,.2f}"

        # Plot
        fig = go.Figure()

        # Shaded confidence interval band
        fig.add_trace(
            go.Scatter(
                x=x + x[::-1],
                y=upper + lower[::-1],
                fill="toself",
                fillcolor="rgba(128,128,128,0.2)",
                line=dict(color="rgba(255,255,255,0)"),
                hoverinfo="skip",
                showlegend=False,
                name="95% CI",
            )
        )

        # Main line with hover text showing CI
        fig.add_trace(
            go.Scatter(
                x=x,
                y=running_means,
                mode="lines",
                line=dict(width=3, color="firebrick"),
                name="Cumulative Avg Error",
                customdata=list(
                    zip(
                        ci,
                    )
                ),
                hovertemplate=(
                    "n=%{x}<br>"
                    "Avg Error=$%{y:,.2f}<br>"
                    "±95% CI=$%{customdata[0]:,.2f}<extra></extra>"
                ),
            )
        )

        fig.update_layout(
            title=title,
            xaxis_title="Number of Datapoints",
            yaxis_title="Error ($)",
            width=800,
            height=300,
            template="plotly_white",
            showlegend=False,
        )

        fig.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        mse = mean_squared_error(self.truths, self.guesses)
        r2 = r2_score(self.truths, self.guesses) * 100
        title = f"{self.title} results<br><b>Error:</b> ${average_error:,.2f} <b>MSE:</b> {mse:,.0f} <b>r²:</b> {r2:.1f}%"
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size)):
            title, guess, truth, error, color = self.run_datapoint(i)
            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        clear_output(wait=True)
        self.report()


def evaluate(function, data, size=DEFAULT_SIZE):
    Tester(function, data, size=size).run()

In [487]:
# evaluate(model_predict_ensemble, test)
evaluate(model_predict_diverse_beam, test)


  0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [ ]:
# median
# temp 0.1 - 111.27
# temp 0.3 - 97.8
# temp 0.5 - 99.58
# temp 0.7 - 90.84
# temp 1.0 - 97.74

# geometric mean
# temp 0.7 - 90.26 (n = 10); 104.3 (n = 5)

# diverse beam search with 10 beams, 5 groups, penalty 1.0  and geometric mean - 86.7
# diverse beam search with 10 beams, 10 groups, penalty 1.5  and geometric mean - 85.84
